# Version 2

- Add log system
- Multiple database
- Update database
- Operators and Datafields Search Tool
- Search tool usage check agent
- (2026.5.9) Add huggingface sync script, for storing docs in huggingface bucket 
- (2026.5.10) Add independent embedding python (work both Windows and Linux)
- (2025.5.10) Fix potential error in `ingest_embeddings`. Add global cache variable to store important content
- (2025.5.10~5.11) Add terminal html colorful log, to store the agent output accurately

In [ ]:
# Temp install command
# %pip install langchain langchain-text-splitters langchain-chroma langchain-openai langchain-community langchain-huggingface crewai crewai-tools sentence-transformers ipywidgets pypdf tqdm ansi2html

In [ ]:
import os
import json
import requests
from crewai import Agent, Task, Crew, Process, LLM
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from crewai.tools import tool
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from config.config import API_KEY_GEMINI, API_KEY_MOONSHOT
import logging
import datetime

In [2]:
def setup_logger(log_dir, log_name, logger_obj_name="logger_obj_name"):
    """
    Configure the logger system: output to both console and file simultaneously.
    """
    # 1. If the log directory doesn't exist, create it
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
        print(f"Log directory created: {log_dir}")

    # 2. Generate a timestamped log filename, e.g., scraper_log_20231027_103000.log
    log_filename = f"{log_name}-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}.log"
    log_filepath = os.path.join(log_dir, log_filename)

    # 3. Create Logger object
    logger = logging.getLogger(logger_obj_name)
    logger.setLevel(logging.INFO) # Set the minimum logger level
    logger.handlers = [] # Clear previous handlers to prevent duplicate printing

    # --- Define a unified format (time accurate to the second) ---
    # %(asctime)s : Time
    # %(levelname)s : Log level (INFO/ERROR)
    # %(message)s : Your message content
    file_formatter = logging.Formatter(
        '[%(asctime)s][%(levelname)s] %(message)s', 
        datefmt="%y-%#m-%#d %H:%M:%S"
    )
    console_formatter = logging.Formatter(
        '[%(asctime)s][%(levelname)s] %(message)s', 
        datefmt="%y-%#m-%#d %H:%M:%S"
    )

    # --- Handler 1: File output (detailed, with timestamp) ---
    file_handler = logging.FileHandler(log_filepath, encoding='utf-8')
    file_handler.setFormatter(file_formatter)
    logger.addHandler(file_handler)

    # --- Handler 2: Console output (concise, for human reading) ---
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(console_formatter)
    logger.addHandler(console_handler)

    logger.info(f"✅ logger System Started")
    logger.info(f"Log file path: {log_filepath}")
    return logger

In [ ]:
# ====================== CONFIG: EVERYTHING ON YOUR OTHER DRIVE ======================
BASE_DIR = "D:/AI_Data/Computer/WorldQuantBrain-Agent/"
WQB_DOCS_PATH = "D:/AI_Data/WQB-Consultant-Data/Docs/"
WQB_FORUM_PATH = "D:/AI_Data/WQB-Consultant-Data/Docs/Forums/"
WQB_OFFICIAL_DOCS_PATH = "D:/AI_Data/WQB-Consultant-Data/Docs/OfficialDocs/"
WQB_PAYMENT_POLICY_PATH = "D:/AI_Data/WQB-Consultant-Data/Docs/PaymentPolicy/"
OPERATOR_FILE_PATH = "D:/AI_Data/WQB-Consultant-Data/Operators/Operators-Agent.json"
DATAFIELDS_FILE_PATH = "D:/AI_Data/WQB-Consultant-Data/DataFields/Datafield-Dataset-Category-Description.json"
CHROMA_DIR = BASE_DIR + "embedding_db/quant_forum_chroma/"
BGEM3_DIR = BASE_DIR + "embedding_db/quant_forum_bgem3/"
# EMBEDDING_DB_DIR = BASE_DIR + "wqb_embedding_db/"
EMBEDDING_DB_DIR = BGEM3_DIR
HF_CACHE_DIR = BASE_DIR + "cache/hf/"
LOG_DIR = BASE_DIR + "logs/" + datetime.datetime.now().strftime("%Y%m") + "/"

# Create the cache folder if it doesn't exist yet
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.makedirs(EMBEDDING_DB_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# ====================== INITIALIZE LOGGER ======================
logger = setup_logger(LOG_DIR, "wqb_agent", "wqb_main_logger")

# Loader Dict
LOADER_DICT = {
    "pdf": PyPDFLoader,
    "txt": TextLoader
}

# ====================== LOAD JSON DATA ======================
with open(OPERATOR_FILE_PATH, 'r', encoding='utf-8') as f:
    operators_data = json.load(f)

with open(DATAFIELDS_FILE_PATH, 'r', encoding='utf-8') as f:
    datafields_data = json.load(f)

[26-4-19 19:53:13][INFO] ✅ logger System Started
[26-4-19 19:53:13][INFO] Log file path: D:/AI_Data/Computer/Agent-WQB/logs/2026-04/wqb_agent-20260419-195313.log


In [ ]:
# Get Model List
# import requests
# from config.config import API_KEY_MOONSHOT

url = "https://api.moonshot.cn/v1/models"
headers = {
    "Authorization": f"Bearer {API_KEY_MOONSHOT}"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    models = response.json()
    for model in models['data']:
        print(f"Model ID: {model['id']}")
else:
    print(f"Error: {response.status_code}, {response.text}")

Model ID: kimi-k2.5
Model ID: moonshot-v1-128k
Model ID: kimi-k2-0905-preview
Model ID: kimi-k2-0711-preview
Model ID: moonshot-v1-auto
Model ID: moonshot-v1-8k-vision-preview
Model ID: moonshot-v1-32k
Model ID: moonshot-v1-8k
Model ID: kimi-k2-thinking-turbo
Model ID: moonshot-v1-128k-vision-preview
Model ID: kimi-k2-thinking
Model ID: kimi-k2-turbo-preview
Model ID: moonshot-v1-32k-vision-preview
Model ID: kimi-k2.6


In [ ]:
# ==================================== CELL: CrewAI LLM (Fixed for your proxy) ====================================

base_url = "http://100.127.232.48:8000/v1"

# Define headers with Bearer Token authentication
headers = {
    "Authorization": f"Bearer {API_KEY_GEMINI}",
    "Content-Type": "application/json"
}

# get available models to verify connection (optional)
response = requests.get(base_url + "/models", headers=headers)
if response.status_code == 200:
    models = response.json()
    print("Available models:", models)
else:
    print("Failed to connect to LLM proxy. Status code:", response.status_code)
    print("Response:", response.text)

Available models: {'object': 'list', 'data': [{'id': 'gemini-3.1-pro', 'object': 'model', 'created': 1776599768, 'owned_by': 'gemini-web'}, {'id': 'gemini-3.0-flash', 'object': 'model', 'created': 1776599768, 'owned_by': 'gemini-web'}, {'id': 'gemini-3.0-flash-thinking', 'object': 'model', 'created': 1776599768, 'owned_by': 'gemini-web'}]}


In [ ]:
# Use your exact proxy settings
llm_gemini_pro = LLM(
    model="gemini-3.1-pro",   # ← change if your proxy uses a different model name
    base_url=base_url,
    api_key=API_KEY_GEMINI,
    temperature=0.4,          # slightly lower = more stable
    max_tokens=8192,
    timeout=180,              # give it more time
    max_retries=5,            # extra retries
)

llm_gemini_flash = LLM(
    model="gemini-3.0-flash-thinking",   # ← change if your proxy uses a different model name
    base_url=base_url,
    api_key=API_KEY_GEMINI,
    temperature=0.6,          # slightly lower = more stable
    max_tokens=8192,
    timeout=180,              # give it more time
    max_retries=5,            # extra retries
)

### Download / Load Embedding Model

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3", # excellent for Chinese
    cache_folder=HF_CACHE_DIR,  # Use the custom cache directory
    model_kwargs={"device": "cpu"},           # force CPU (your low GPU setup)
    encode_kwargs={"normalize_embeddings": True},  # best for Chroma similarity search
    show_progress=True
)

### Embedding (Multiple Database)

- Forums
- Official Docs
- Payment Policy

In [ ]:
# ==================================== Ingest PDFs (ONE-TIME) ====================================
def ingest_embeddings(docs_path, embeddings, vectorstore_dir, docs_type, source_type=""):
    logger.info(f"Starting {source_type} ingestion from: {docs_path}")

    if docs_type not in LOADER_DICT:
        logger.error(f"Unsupported document type: {docs_type}. Supported types are: {list(LOADER_DICT.keys())}")
        return False

    docs_path_abs = os.path.abspath(docs_path)
    vectorstore_dir_abs = os.path.abspath(vectorstore_dir)
    ingest_state_path = os.path.join(docs_path_abs, "ingested_files.json")

    # State is tied to vectorstore_dir. If DB path changes, treat as a fresh DB.
    ingest_state = {
        "vectorstore_dir": vectorstore_dir_abs,
        "files": []
    }

    if os.path.exists(ingest_state_path):
        try:
            with open(ingest_state_path, "r", encoding="utf-8") as f:
                existing_state = json.load(f)
            if isinstance(existing_state, dict) and existing_state.get("vectorstore_dir") == vectorstore_dir_abs:
                ingest_state = {
                    "vectorstore_dir": vectorstore_dir_abs,
                    "files": existing_state.get("files", []) if isinstance(existing_state.get("files", []), list) else []
                }
            else:
                logger.info("Detected a new embedding DB path. Resetting ingest tracking state.")
        except Exception as e:
            logger.warning(f"Failed to read ingest state file, starting fresh. Reason: {e}")

    already_ingested = set(ingest_state["files"])

    loader = DirectoryLoader(
        docs_path,
        glob=f"**/*.{docs_type}",
        loader_cls=LOADER_DICT.get(docs_type),
        silent_errors=False,
        show_progress=True
    )

    logger.info("Loading documents (this may take a while)...")
    try:
        docs = loader.load()
        logger.info(f"Successfully loaded {len(docs)} {docs_type.upper()} files.")
    except Exception as e:
        logger.error(f"❌ Failed to load {docs_type.upper()}s: {e}", exc_info=True)
        return False

    docs_to_ingest = []
    newly_added_files = set()
    # for doc in docs:
    #     source_file = os.path.abspath(doc.metadata.get("source", ""))
    #     if source_file and source_file in already_ingested:
    #         continue
    #     docs_to_ingest.append(doc)
    #     if source_file:
    #         newly_added_files.add(source_file)

    # ============ Fix: Use relative path record ===============
    for doc in docs:
        raw_source = doc.metadata.get("source", "")
        
        if raw_source:
            # 1. Get the absolute path of the file
            abs_source = os.path.abspath(raw_source)
            
            # 2. Find the relative path from your project root (BASE_DIR)
            # This turns '/data/Docs/OfficialDocs/file.pdf' into 'Docs/OfficialDocs/file.pdf'
            # Docs/OfficialDocs/file.pdf is better than /Docs/OfficialDocs/file.pdf
            rel_source = os.path.relpath(abs_source, start=BASE_DIR)
            
            # 3. Force forward slashes for Windows/Linux cross-compatibility
            source_file = rel_source.replace("\\", "/")
        else:
            source_file = ""
            
        if source_file and source_file in already_ingested:
            continue
            
        docs_to_ingest.append(doc)
        
        if source_file:
            newly_added_files.add(source_file)
    
    if not docs_to_ingest:
        logger.info("No new files to ingest. All matching files already exist in ingest state.")
        return True

    logger.info(f"New files to ingest: {len(docs_to_ingest)}")

    logger.info("Splitting documents into chunks...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400)
    splits = text_splitter.split_documents(docs_to_ingest)
    logger.info(f"Created {len(splits)} text chunks.")

    # Add source metadata to each chunk for better traceability if source_type is provided
    if source_type:
        for chunk in splits:
            chunk.metadata["source_type"] = source_type
        logger.info(f"Added source_type metadata to each chunk: {source_type}")

    logger.info("Initializing Chroma vectorstore and generating embeddings...")
    try:
        Chroma.from_documents(
            documents=splits,
            embedding=embeddings,
            persist_directory=vectorstore_dir
        )

        updated_files = sorted(already_ingested.union(newly_added_files))
        with open(ingest_state_path, "w", encoding="utf-8") as f:
            json.dump({"vectorstore_dir": vectorstore_dir_abs, "files": updated_files}, f, ensure_ascii=False, indent=2)

        logger.info(f"✅ Successfully ingested {len(splits)} chunks into Chroma DB at {vectorstore_dir} for source: {source_type}")
        logger.info(f"Updated ingest tracking file: {ingest_state_path}")
        return True
    except Exception as e:
        logger.error(f"❌ Error during Chroma DB creation: {e}", exc_info=True)
        raise e

In [ ]:
# First run only (First run forum data then official docs)
# Success_Forum = ingest_embeddings(WQB_FORUM_PATH, embeddings, EMBEDDING_DB_DIR, "pdf", "forum_pdf")
# Success_OfficialDocs = ingest_embeddings(WQB_OFFICIAL_DOCS_PATH, embeddings, EMBEDDING_DB_DIR, "pdf", "official_docs_pdf")
# Success_PaymentPolicy = ingest_embeddings(WQB_PAYMENT_POLICY_PATH, embeddings, EMBEDDING_DB_DIR, "pdf", "payment_policy_pdf")

In [7]:
# ==================================== Initialize Retriever (for querying) ====================================
vectorstore = Chroma(persist_directory=EMBEDDING_DB_DIR, embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

### Search Tool

In [16]:
# ====================== DOCS SEARCH TOOL ======================
@tool("retrieve_text_data")
def retrieve_text_data(query: str) -> str:
    """Fetches relevant text snippets based on a query.
    Input a highly specific financial concept or math operator (e.g., 'supply chain momentum', 'analyst revision').
    Returns text context to be used for answering user queries."""
    
    docs = retriever.invoke(query)
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

# ====================== JSON SEARCH TOOLS ======================
@tool("search_operators")
def search_operators(query: str) -> str:
    """Search the operator definitions and syntax. 
    Input a concept like 'neutralize', 'moving average', or 'rank'."""
    results = []
    query_lower = query.lower()
    for op_name, op_details in operators_data.items():
        if (query_lower in op_name.lower() or 
            query_lower in op_details.get('description', '').lower() or 
            query_lower in op_details.get('category', '').lower()):
            
            res = f"Operator: {op_name}\nSyntax: {op_details.get('definition')}\nDesc: {op_details.get('description')}"
            results.append(res)
            
        if len(results) >= 10:  # Limit results to save context window
            break
            
    return "\n---\n".join(results) if results else "No matching operators found."

@tool("search_datafields")
def search_datafields(query: str) -> str:
    """Search the data dictionary for dataset fields. 
    Input a concept like 'pollution', 'inventory', 'analyst', or 'price'."""
    results = []
    query_lower = query.lower()
    for field_name, field_details in datafields_data.items():
        if (query_lower in field_name.lower() or 
            query_lower in field_details.get('description', '').lower() or 
            query_lower in field_details.get('category_name', '').lower()): # Note: matching your JSON typo 'category_name'
            
            res = f"Field: {field_name}\nType: {field_details.get('type')}\nDesc: {field_details.get('description')}"
            results.append(res)
            
        if len(results) >= 15:  # Limit results to save context window
            break
            
    return "\n---\n".join(results) if results else "No matching data fields found."


# ====================== WQB SIMULATE API TOOL ======================
WQB_SIMULATE_API_URL = os.getenv("WQB_SIMULATE_API_URL", "").strip()
WQB_SIMULATE_TIMEOUT_SECONDS = int(os.getenv("WQB_SIMULATE_TIMEOUT_SECONDS", "120"))


@tool("wqb_simulate_api")
def wqb_simulate_api(settings: str, regular_formula: str) -> str:
    """Run alpha simulation by calling your WQB simulate API.

    Input:
    - settings: JSON string of simulator settings.
    - regular_formula: WorldQuant regular formula string.
    """
    if not WQB_SIMULATE_API_URL:
        return (
            "Simulation API URL is not configured. "
            "Set environment variable WQB_SIMULATE_API_URL first."
        )

    try:
        settings_payload = json.loads(settings) if isinstance(settings, str) else settings
    except json.JSONDecodeError as e:
        return f"Invalid settings JSON: {e}"

    payload = {
        "settings": settings_payload,
        "regular_formula": regular_formula,
    }

    try:
        response = requests.post(
            WQB_SIMULATE_API_URL,
            json=payload,
            timeout=WQB_SIMULATE_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        try:
            return json.dumps(response.json(), ensure_ascii=False)
        except Exception:
            return response.text
    except Exception as e:
        return f"Simulation API request failed: {e}"

In [ ]:
# ==================================== DEBUG: Test if your docs PDFs are loaded ====================================
print("Testing retrieve_text_data tool...")

test_result = retrieve_text_data.run("AllRightsReserved")

print("\n" + "="*80)
print("TOOL TEST RESULT:")
print(test_result)  # first 1500 chars
print("\n" + "="*80)
print(f"Length of returned text: {len(test_result)} characters")

In [ ]:
# ==================================== DEBUG: Test JSON search tools ====================================
print("Testing search_operators tool...")
test_op_result = search_operators.run("neutralize")
print("\n" + "="*80)
print("Search_Operators TOOL TEST RESULT:")
print(test_op_result)
print("\n" + "="*80)

In [ ]:
print("Testing search_datafields tool...")
test_df_result = search_datafields.run("health")
print("\n" + "="*80)
print("Search_DataFields TOOL TEST RESULT:")
print(test_df_result)
print("\n" + "="*80)

### Test Agents (Test whether the search tool is usable)

In [ ]:
from wqbquant_searchtool_test import test_agents
test_agents(retrieve_text_data, search_operators, search_datafields, llm_gemini_flash)

### Agents & Crew

In [ ]:
# ====================== AGENTS (Your Quant Research Team) ======================
researcher = Agent(
    role="WorldQuant Docs Researcher & Master Analyst",
    goal="Use the `retrieve_text_data` tool to fetch context for the user's request. Base your output on the tool's results. Never answer from general knowledge.",
    backstory="""You are a veteran WorldQuant Brain consultant. You are an advanced AI agent equipped with a local vector database interface.
    You do not have the PDFs in your internal memory; you rely ENTIRELY on the `retrieve_text_data` tool. You use lateral thinking. 
    If a user asks about 'volume', you search for 'liquidity shock', 'turnover spike', or 'institutional block trades'.
    You always call the `retrieve_text_data` tool multiple times with completely different vocabulary each time to get a full picture.
    """,
    tools=[retrieve_text_data],
    llm=llm_gemini_pro,
    verbose=True,
    allow_delegation=False
)

ideator = Agent(
    role="Low-Correlation BRAIN Innovator",
    goal="Create truly innovative, submittable alphas using specific alternative data fields.",
    backstory="""You are a contrarian quant. You MUST use the `search_datafields` tool to find real, specific dataset names (like 'anti_pollution_policy_industry_rank') to build your hypotheses. Do not hallucinate data field names.""",
    tools=[search_datafields], # <--- ADDED TOOL
    llm=llm_gemini_pro,
    verbose=True
)

coder = Agent(
    role="WorldQuant BRAIN Expression Expert",
    goal="Convert the idea into a valid expression using exact Operator syntax and Exact Data fields.",
    backstory="""You are an ex-WorldQuant Brain coder. 
    1. You MUST use `search_operators` to check the exact syntax of functions (e.g., checking if add() takes a filter argument). 
    2. You MUST use `search_datafields` to ensure you are using the exact string name for data sets.
    You never guess operator syntax.""",
    tools=[search_operators, search_datafields], # <--- ADDED TOOLS
    llm=llm_gemini_flash,
    verbose=True
)

validator = Agent(
    role="WorldQuant Submission Validator",
    goal="Ensure the alpha is innovative, low-correlation, and ready to submit. Output ONLY in the exact user-specified format.",
    backstory="You are the final gatekeeper. You check for simulator compatibility, low correlation, and economic soundness. You never add extra text outside the required format.",
    tools=[wqb_simulate_api],
    llm=llm_gemini_pro,
    verbose=True
)

# ====================== TASKS & CREW ======================
task1 = Task(
    description="""
    Based on the user's request, you are authorized and required to use the `retrieve_text_data` tool.
    
    STEP 1: Brainstorm 3 different, highly specific keyword phrases related to the request.
    STEP 2: Call the `retrieve_text_data` tool using one of your brainstormed phrases. If the result is poor, call it again with a different phrase.
    STEP 3: Synthesize the returned database snippets.
    
    Focus on extracting real opinions and specific discussions.
    """,
    expected_output="Structured summary of the retrieved database content with direct quotes.",
    agent=researcher
)

task2 = Task(
    description="""
    Generate 3-5 genuinely innovative alpha ideas.
    CRITICAL: You MUST use the `search_datafields` tool to search for keywords related to your ideas (e.g., "ESG", "Analyst", "Supply Chain") and include the EXACT field names in your output.
    Focus on low correlation and economic rationale.
    """,
    expected_output="Numbered list of 3-5 alpha ideas containing exact Data Field names, hypothesis, and low-correlation justification.",
    agent=ideator
)

task3 = Task(
    description="""
    Take the BEST idea from Task 2 and write a clean, valid WorldQuant BRAIN expression.
    CRITICAL: You MUST use the `search_operators` tool to verify the syntax of every math/logic function you plan to use before writing the final expression.
    Choose realistic Target Settings (Region, Universe, Neutralization, Delay, Decay, Truncation).
    """,
    expected_output="One complete alpha in the exact user format (Alpha Name + Economic Hypothesis + Target Settings + Full BRAIN Expression).",
    agent=coder
)

task4 = Task(
    description="""
    Act as strict WorldQuant reviewer.
    Critique the alpha from Task 3 for innovation, low correlation, simulator compatibility, and economic sense.
    If needed, improve it slightly.

    CRITICAL: Before finalizing, call `wqb_simulate_api` exactly once with:
    - settings: a JSON string that includes Region, Universe, Neutralization, Delay, Decay, and Truncation.
    - regular_formula: the final formula string.

    Use simulation output to decide whether the alpha is good enough; if not, improve the formula once and return the improved one.

    THEN output ONLY the final alpha in the EXACT format the user wants:
    
    **Alpha Name:** ...
    **Economic Hypothesis:** ...
    **Target Settings:** Region: ___ | Universe: ___ | Neutralization: ___ | Delay: ___ | Decay: ___ | Truncation: ___
    **Full BRAIN Expression:** ...
    
    Do not add any extra explanation or text outside this format.
    """,
    expected_output="Final alpha in the exact markdown format requested by the user.",
    agent=validator
)

crew = Crew(
    agents=[researcher, ideator, coder, validator],
    tasks=[task1, task2, task3, task4],
    process=Process.sequential,
    verbose=True,
    # tracing=True
)

In [ ]:
# ====================== RUN ======================
if __name__ == "__main__":
    result = crew.kickoff(inputs={
    "user_request": """
    Explore forum discussions specifically around Analyst Estimates (EPS, Revisions) or Supply Chain inventory data. 
    Find out what basic combinations people are using, and generate alphas that apply non-linear operators 
    (like sign, abs, or conditional logic) to these specific datasets to achieve low correlation.
    """.strip()
    })
    print(result)